In [1]:
import numpy as np
import pandas as pd

In [3]:
czetyl = pd.read_csv('czetyl.tsv', sep='\t')
paraczech = pd.read_csv('../final-paraczech-package/paraczech/paraczech_forms.csv')

In [30]:
df = paraczech[['lexeme', 'phon_form', 'candidate_phon_form']].drop_duplicates(subset=['lexeme']).copy()

m_phon_filled = df['phon_form'].notna() & (df['phon_form'] != '#MISSING#')
m_phon_cand_filled = df['candidate_phon_form'].notna() & (df['candidate_phon_form'] != '')

conditions = [m_phon_filled, m_phon_cand_filled]
choices = ['native', 'loan']
df['our_origin'] = np.select(conditions, choices, default='unknown')

df_final = df[['lexeme', 'our_origin']]

In [32]:
df_final['czetyl_origin'] = df_final['lexeme'].map(czetyl.set_index('word')['is_loan'])

C:\Users\lukas\AppData\Local\Temp\ipykernel_24552\2696762295.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['czetyl_origin'] = df_final['lexeme'].map(czetyl.set_index('word')['is_loan'])


In [33]:
tp = df_final[(df_final['our_origin'] == 'loan') & (df_final['czetyl_origin'] == 'loan')].shape[0]
fp = df_final[(df_final['our_origin'] == 'loan') & (df_final['czetyl_origin'] == 'native')].shape[0]
fn = df_final[(df_final['our_origin'] == 'native') & (df_final['czetyl_origin'] == 'loan')].shape[0]
tn = df_final[(df_final['our_origin'] == 'native') & (df_final['czetyl_origin'] == 'native')].shape[0]

In [43]:
df_final['czetyl_origin'].value_counts()

czetyl_origin
loan      5844
native    3297
Name: count, dtype: int64

In [41]:
acc = (tp+tn)/(tp+tn+fp+fn)
print(f'Accuracy={acc}')

rec = tp/(tp+fn)
print(f'Recall={rec}')

pre = tp/(tp+fp)
print(f'Precision={pre}')

f1s = 2*((pre*rec)/(pre+rec))
print(f'F1-score={f1s}')

Accuracy=0.8199321737227875
Recall=0.7375085557837098
Precision=0.9746720940750792
F1-score=0.8396649133060589
